<a href="https://colab.research.google.com/github/Sushit-prog/GenAI/blob/main/weatheragent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-mistralai langchain-community tavily-python requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00


In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_mistralai import ChatMistralAI
from tavily import TavilyClient
import requests
import os

In [ ]:
from getpass import getpass
os.environ["MISTRAL_API_KEY"] = getpass("Mistral API key: ")
os.environ["OPENWEATHER_API_KEY"] = getpass("OpenWeather API key: ")
os.environ["TAVILY_API_KEY"]      = getpass("Tavily API key: ")

print("Key set.")


Mistral API key: ··········
OpenWeather API key: ··········
Tavily API key: ··········
Key set.


In [ ]:
# Creating tool
@tool
def get_weather(city: str) -> str:
    """Fetches live temperature and weather conditions for a city."""
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city,
        "appid": os.environ["OPENWEATHER_API_KEY"],
        "units": "metric"
    }
    response = requests.get(url, params=params, timeout=10)
    data = response.json()

    if response.status_code == 404:
        return f"City '{city}' not found."

    return (
        f"Weather in {data['name']}, {data['sys']['country']}:\n"
        f"  🌡️ Temp      : {data['main']['temp']}°C (feels like {data['main']['feels_like']}°C)\n"
        f"  💧 Humidity  : {data['main']['humidity']}%\n"
        f"  🌬️ Wind      : {data['wind']['speed']} m/s\n"
        f"  ☁️ Condition : {data['weather'][0]['description'].capitalize()}"
    )

In [ ]:
# News Tool

@tool
def get_city_news(city: str) -> str:
    """Fetches latest news headlines for a city using Tavily search."""
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    results = client.search(
        query=f"Latest news in {city} today",
        search_depth="basic",
        max_results=4
    )

    if not results.get("results"):
        return f"No news found for '{city}'."

    lines = [f"📰 News for {city}:\n"]
    for i, item in enumerate(results["results"], 1):
        lines.append(
            f"{i}. {item.get('title', 'No title')}\n"
            f"   🔗 {item.get('url', '')}\n"
            f"   {item.get('content', '')[:200]}..."
        )
    return "\n\n".join(lines)

In [ ]:
# tool binding
llm = ChatMistralAI(model="mistral-large-latest", temperature=0.2)

tools = [get_weather, get_city_news]
tool_map = {t.name: t for t in tools}

llm_with_tools = llm.bind_tools(tools)

print(f"✅ LLM ready. Tools: {list(tool_map.keys())}")

✅ LLM ready. Tools: ['get_weather', 'get_city_news']


In [ ]:
def human_approval_gate(tool_calls):
    print("\n" + "═" * 50)
    print("  🔔 Tool Execution Request — Approval Required")
    print("═" * 50)

    for call in tool_calls:
        icon = "🌤️" if call["name"] == "get_weather" else "📰"
        print(f"  {icon}  Tool : {call['name']}")
        print(f"       Args : {call['args']}")
        print("─" * 50)

    while True:
        ans = input("  Approve? [y/n]: ").strip().lower()
        if ans in ("y", "yes"):
            print("  ▶ Approved.\n")
            return True
        if ans in ("n", "no"):
            print("  ✋ Denied.\n")
            return False
        print("  Please type y or n.")

In [ ]:
def run_agent_step(conversation):
    ai_msg = llm_with_tools.invoke(conversation)

    if not ai_msg.tool_calls:
        return ai_msg.content

    if not human_approval_gate(ai_msg.tool_calls):
        return "Tool execution cancelled. Try again!"

    conversation.append(ai_msg)

    for call in ai_msg.tool_calls:
        print(f"  ⚙️ Running {call['name']}({call['args']})...")
        result = tool_map[call["name"]].invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return llm_with_tools.invoke(conversation).content

In [ ]:
SYSTEM_PROMPT = """You are a helpful City Intelligence Assistant.
You have two tools:
- get_weather: for weather, temperature, humidity questions about a city
- get_city_news: for news, events, recent happenings in a city

Always extract the city name before calling a tool.
If the query covers both weather AND news, call both tools."""

conversation = [SystemMessage(content=SYSTEM_PROMPT)]

print("🌆 City Intelligence Agent ready! Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("👋 Bye!")
        break

    conversation.append(HumanMessage(content=user_input))

    try:
        response = run_agent_step(conversation)
    except Exception as e:
        response = f"❌ Error: {e}"

    print(f"\n🤖 Agent:\n{response}\n")
    print("─" * 50)

    conversation.append(AIMessage(content=response))

🌆 City Intelligence Agent ready! Type 'quit' to exit.

You: what is the weather in london

══════════════════════════════════════════════════
  🔔 Tool Execution Request — Approval Required
══════════════════════════════════════════════════
  🌤️  Tool : get_weather
       Args : {'city': 'London'}
──────────────────────────────────────────────────
  Approve? [y/n]: y
  ▶ Approved.

  ⚙️ Running get_weather({'city': 'London'})...

🤖 Agent:
Here’s the current weather in **London**:

- **Temperature**: **10.42°C** (feels like **9.01°C**)
- **Humidity**: **57%**
- **Wind**: **0.45 m/s** (light breeze)
- **Conditions**: **Clear sky** ☀️

Enjoy the pleasant weather! Let me know if you'd like a forecast or news updates.

──────────────────────────────────────────────────
You: what is the weather in Sodepur

══════════════════════════════════════════════════
  🔔 Tool Execution Request — Approval Required
══════════════════════════════════════════════════
  🌤️  Tool : get_weather
       Args : {

In [ ]:
import os
key = os.environ.get("MISTRAL_API_KEY", "NOT SET")
print(f"Key starts with: {key[:8]}...")
print(f"Key length: {len(key)}")

Key starts with: hf_CKeDy...
Key length: 37
